# SFR-002: 장애 발생 전 징후 포착 시스템

## 목표
- 시계열 데이터 기반 이상 징후 탐지
- 복수 데이터 비교를 통한 이상 징후 판단
- LLM을 활용한 이상 징후 의미 및 잠재 영향 설명

## 1. 환경 설정 및 라이브러리 로딩

In [ ]:
# 필요한 라이브러리 설치 (최초 1회)
# !pip install pandas numpy scipy scikit-learn matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로딩 완료!")

## 2. 데이터 로딩

In [ ]:
# 경로 설정
DATA_PATH = './output/'

# 주요 데이터 로딩
print("데이터 로딩 중...")

# NMS 데이터
df_nms_master = pd.read_csv(f'{DATA_PATH}01_NMS_장비마스터.csv', encoding='utf-8-sig')
df_nms_dev_perf = pd.read_csv(f'{DATA_PATH}02_NMS_장비성능_5min.csv', encoding='utf-8-sig')
df_nms_dev_last = pd.read_csv(f'{DATA_PATH}03_NMS_장비성능_last.csv', encoding='utf-8-sig')
df_nms_if_perf = pd.read_csv(f'{DATA_PATH}04_NMS_IF성능_5min.csv', encoding='utf-8-sig')
df_nms_if_last = pd.read_csv(f'{DATA_PATH}05_NMS_IF성능_last.csv', encoding='utf-8-sig')

# SMS 데이터
df_sms_master = pd.read_csv(f'{DATA_PATH}08_SMS_장비마스터.csv', encoding='utf-8-sig')
df_sms_cpu = pd.read_csv(f'{DATA_PATH}10_SMS_CPU_5min.csv', encoding='utf-8-sig')
df_sms_memory = pd.read_csv(f'{DATA_PATH}11_SMS_메모리_5min.csv', encoding='utf-8-sig')
df_sms_filesystem = pd.read_csv(f'{DATA_PATH}12_SMS_파일시스템_5min.csv', encoding='utf-8-sig')
df_sms_network = pd.read_csv(f'{DATA_PATH}13_SMS_네트워크_5min.csv', encoding='utf-8-sig')

print("데이터 로딩 완료!")
print(f"\n=== 데이터 크기 ===")
print(f"NMS 장비 마스터: {len(df_nms_master):,}건")
print(f"NMS 장비 성능 5분: {len(df_nms_dev_perf):,}건")
print(f"NMS IF 성능 5분: {len(df_nms_if_perf):,}건")
print(f"SMS 서버 마스터: {len(df_sms_master):,}건")
print(f"SMS CPU 5분: {len(df_sms_cpu):,}건")
print(f"SMS 메모리 5분: {len(df_sms_memory):,}건")

## 3. 데이터 전처리

In [ ]:
def convert_ymdhms_to_datetime(df, ymdhms_col='YMDHMS'):
    """
    YMDHMS (예: 20260310143000) 컬럼을 datetime으로 변환
    """
    df = df.copy()
    df['DATETIME'] = pd.to_datetime(df[ymdhms_col].astype(str), format='%Y%m%d%H%M%S', errors='coerce')
    return df

# 시간 변환 적용
df_nms_dev_perf = convert_ymdhms_to_datetime(df_nms_dev_perf)
df_nms_if_perf = convert_ymdhms_to_datetime(df_nms_if_perf)
df_sms_cpu = convert_ymdhms_to_datetime(df_sms_cpu)
df_sms_memory = convert_ymdhms_to_datetime(df_sms_memory)

print("시간 변환 완료!")
print(f"\n데이터 기간: {df_nms_if_perf['DATETIME'].min()} ~ {df_nms_if_perf['DATETIME'].max()}")

In [ ]:
# 장비 마스터에서 주요 정보 추출
def get_device_info(master_df):
    """
    장비 마스터에서 분석에 필요한 주요 정보만 추출
    """
    cols = ['MNG_NO', 'DEV_NAME', 'USER_DEV_NAME', 'DEV_IP', 'DEV_KIND1', 'DEV_KIND2', 
            'VENDOR', 'MODEL', 'PARENT_MNG_NO']
    available_cols = [c for c in cols if c in master_df.columns]
    return master_df[available_cols].copy()

device_info_nms = get_device_info(df_nms_master)
device_info_sms = get_device_info(df_sms_master)

print("NMS 장비 정보:")
display(device_info_nms.head())
print(f"\nSMS 서버 정보:")
display(device_info_sms.head())

## 4. 기초 탐색 분석 (EDA)

In [ ]:
# NMS IF 성능 데이터 기초 통계
print("=== NMS IF 성능 기초 통계 ===")
perf_cols = ['AVG_INBPS', 'AVG_OUTBPS', 'AVG_INERR', 'AVG_OUTERR', 'INBPS_RATE', 'OUTBPS_RATE']
available_perf_cols = [c for c in perf_cols if c in df_nms_if_perf.columns]
display(df_nms_if_perf[available_perf_cols].describe())

In [ ]:
# SMS CPU 사용률 분포
print("=== SMS CPU 사용률 분포 ===")

# CPU 사용률 계산 (100 - IDLE = 사용률)
if 'IDLE_PCT_AVG' in df_sms_cpu.columns:
    df_sms_cpu['CPU_USAGE_AVG'] = 100 - df_sms_cpu['IDLE_PCT_AVG']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 히스토그램
    axes[0].hist(df_sms_cpu['CPU_USAGE_AVG'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('CPU 사용률 (%)')
    axes[0].set_ylabel('빈도')
    axes[0].set_title('SMS CPU 사용률 분포')
    axes[0].axvline(x=80, color='r', linestyle='--', label='경고 임계값 (80%)')
    axes[0].axvline(x=90, color='darkred', linestyle='--', label='위험 임계값 (90%)')
    axes[0].legend()
    
    # 박스플롯 (서버별)
    df_sms_cpu.boxplot(column='CPU_USAGE_AVG', by='MNG_NO', ax=axes[1])
    axes[1].set_xlabel('서버 번호 (MNG_NO)')
    axes[1].set_ylabel('CPU 사용률 (%)')
    axes[1].set_title('서버별 CPU 사용률 분포')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 메모리 사용률 분포
print("=== SMS 메모리 사용률 분포 ===")

if 'PHYSICAL_USED_PCT' in df_sms_memory.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 히스토그램
    axes[0].hist(df_sms_memory['PHYSICAL_USED_PCT'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
    axes[0].set_xlabel('메모리 사용률 (%)')
    axes[0].set_ylabel('빈도')
    axes[0].set_title('SMS 메모리 사용률 분포')
    axes[0].axvline(x=80, color='orange', linestyle='--', label='경고 임계값 (80%)')
    axes[0].axvline(x=90, color='red', linestyle='--', label='위험 임계값 (90%)')
    axes[0].legend()
    
    # 박스플롯 (서버별)
    df_sms_memory.boxplot(column='PHYSICAL_USED_PCT', by='MNG_NO', ax=axes[1])
    axes[1].set_xlabel('서버 번호 (MNG_NO)')
    axes[1].set_ylabel('메모리 사용률 (%)')
    axes[1].set_title('서버별 메모리 사용률 분포')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()

## 5. 이상 탐지 알고리즘 구현

### 5.1 통계적 이상 탐지 (Z-Score, IQR)

In [ ]:
class StatisticalAnomalyDetector:
    """
    통계적 방법을 이용한 이상 탐지 클래스
    """
    
    def __init__(self, z_threshold=3.0, iqr_multiplier=1.5):
        self.z_threshold = z_threshold
        self.iqr_multiplier = iqr_multiplier
        self.stats = {}
    
    def fit(self, df, group_col, value_col):
        """
        그룹별 통계 계산 (평균, 표준편차, Q1, Q3)
        """
        self.stats = df.groupby(group_col)[value_col].agg([
            'mean', 'std', 
            lambda x: x.quantile(0.25),  # Q1
            lambda x: x.quantile(0.75)   # Q3
        ]).rename(columns={'<lambda_0>': 'Q1', '<lambda_1>': 'Q3'})
        self.stats['IQR'] = self.stats['Q3'] - self.stats['Q1']
        self.stats['lower_bound'] = self.stats['Q1'] - self.iqr_multiplier * self.stats['IQR']
        self.stats['upper_bound'] = self.stats['Q3'] + self.iqr_multiplier * self.stats['IQR']
        return self
    
    def detect_zscore(self, df, group_col, value_col):
        """
        Z-Score 기반 이상 탐지
        """
        df = df.copy()
        df = df.merge(self.stats[['mean', 'std']], left_on=group_col, right_index=True, how='left')
        df['z_score'] = (df[value_col] - df['mean']) / df['std'].replace(0, np.nan)
        df['is_anomaly_zscore'] = df['z_score'].abs() > self.z_threshold
        return df
    
    def detect_iqr(self, df, group_col, value_col):
        """
        IQR 기반 이상 탐지
        """
        df = df.copy()
        df = df.merge(self.stats[['lower_bound', 'upper_bound']], left_on=group_col, right_index=True, how='left')
        df['is_anomaly_iqr'] = (df[value_col] < df['lower_bound']) | (df[value_col] > df['upper_bound'])
        return df
    
    def detect_threshold(self, df, value_col, warning_threshold, critical_threshold):
        """
        고정 임계값 기반 이상 탐지
        """
        df = df.copy()
        df['severity'] = 'normal'
        df.loc[df[value_col] >= warning_threshold, 'severity'] = 'warning'
        df.loc[df[value_col] >= critical_threshold, 'severity'] = 'critical'
        df['is_anomaly_threshold'] = df['severity'] != 'normal'
        return df

print("StatisticalAnomalyDetector 클래스 정의 완료!")

### 5.2 급격한 변화율 탐지

In [ ]:
class ChangeRateDetector:
    """
    급격한 변화율 탐지 클래스
    """
    
    def __init__(self, rate_threshold=0.5, window_size=3):
        """
        rate_threshold: 변화율 임계값 (0.5 = 50% 변화)
        window_size: 이동평균 윈도우 크기
        """
        self.rate_threshold = rate_threshold
        self.window_size = window_size
    
    def detect(self, df, group_col, time_col, value_col):
        """
        그룹별 시간순 정렬 후 변화율 계산
        """
        df = df.copy().sort_values([group_col, time_col])
        
        # 이전 값 대비 변화율 계산
        df['prev_value'] = df.groupby(group_col)[value_col].shift(1)
        df['change_rate'] = (df[value_col] - df['prev_value']) / df['prev_value'].replace(0, np.nan)
        
        # 이동평균 계산
        df['rolling_mean'] = df.groupby(group_col)[value_col].transform(
            lambda x: x.rolling(window=self.window_size, min_periods=1).mean()
        )
        
        # 이동평균 대비 편차율
        df['deviation_from_ma'] = (df[value_col] - df['rolling_mean']) / df['rolling_mean'].replace(0, np.nan)
        
        # 이상 여부 판단
        df['is_spike'] = df['change_rate'].abs() > self.rate_threshold
        df['is_anomaly_change'] = df['is_spike'] | (df['deviation_from_ma'].abs() > self.rate_threshold)
        
        return df

print("ChangeRateDetector 클래스 정의 완료!")

### 5.3 Isolation Forest 기반 이상 탐지

In [ ]:
class MLAnomalyDetector:
    """
    머신러닝 기반 이상 탐지 (Isolation Forest)
    """
    
    def __init__(self, contamination=0.05):
        """
        contamination: 예상 이상치 비율 (기본 5%)
        """
        self.contamination = contamination
        self.model = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)
        self.scaler = StandardScaler()
    
    def fit_predict(self, df, feature_cols):
        """
        복수 피처를 이용한 이상 탐지
        """
        df = df.copy()
        
        # 결측치 처리
        features = df[feature_cols].fillna(df[feature_cols].median())
        
        # 스케일링
        features_scaled = self.scaler.fit_transform(features)
        
        # 이상 탐지 (-1: 이상, 1: 정상)
        predictions = self.model.fit_predict(features_scaled)
        df['is_anomaly_ml'] = predictions == -1
        df['anomaly_score'] = self.model.decision_function(features_scaled)
        
        return df

print("MLAnomalyDetector 클래스 정의 완료!")

## 6. 이상 탐지 실행

### 6.1 SMS CPU 이상 탐지

In [ ]:
# CPU 사용률 이상 탐지
print("=== SMS CPU 이상 탐지 ===")

if 'CPU_USAGE_AVG' in df_sms_cpu.columns:
    # 1. 통계적 이상 탐지
    stat_detector = StatisticalAnomalyDetector(z_threshold=3.0)
    stat_detector.fit(df_sms_cpu, 'MNG_NO', 'CPU_USAGE_AVG')
    
    df_cpu_anomaly = stat_detector.detect_zscore(df_sms_cpu, 'MNG_NO', 'CPU_USAGE_AVG')
    df_cpu_anomaly = stat_detector.detect_iqr(df_cpu_anomaly, 'MNG_NO', 'CPU_USAGE_AVG')
    df_cpu_anomaly = stat_detector.detect_threshold(df_cpu_anomaly, 'CPU_USAGE_AVG', 
                                                     warning_threshold=80, critical_threshold=90)
    
    # 2. 변화율 탐지
    change_detector = ChangeRateDetector(rate_threshold=0.3)
    df_cpu_anomaly = change_detector.detect(df_cpu_anomaly, 'MNG_NO', 'DATETIME', 'CPU_USAGE_AVG')
    
    # 종합 이상 여부
    df_cpu_anomaly['is_anomaly'] = (
        df_cpu_anomaly['is_anomaly_zscore'] | 
        df_cpu_anomaly['is_anomaly_iqr'] | 
        df_cpu_anomaly['is_anomaly_threshold'] |
        df_cpu_anomaly['is_anomaly_change']
    )
    
    # 결과 요약
    anomaly_count = df_cpu_anomaly['is_anomaly'].sum()
    total_count = len(df_cpu_anomaly)
    print(f"\n총 데이터: {total_count:,}건")
    print(f"이상 탐지: {anomaly_count:,}건 ({anomaly_count/total_count*100:.2f}%)")
    print(f"\n이상 유형별 건수:")
    print(f"  - Z-Score 이상: {df_cpu_anomaly['is_anomaly_zscore'].sum():,}건")
    print(f"  - IQR 이상: {df_cpu_anomaly['is_anomaly_iqr'].sum():,}건")
    print(f"  - 임계값 초과: {df_cpu_anomaly['is_anomaly_threshold'].sum():,}건")
    print(f"  - 급격한 변화: {df_cpu_anomaly['is_anomaly_change'].sum():,}건")

In [ ]:
# 이상 탐지 결과 시각화
if 'is_anomaly' in df_cpu_anomaly.columns:
    # 특정 서버의 시계열 시각화
    sample_mng_no = df_cpu_anomaly['MNG_NO'].unique()[0]
    sample_data = df_cpu_anomaly[df_cpu_anomaly['MNG_NO'] == sample_mng_no].copy()
    
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # 정상 데이터
    normal = sample_data[~sample_data['is_anomaly']]
    ax.plot(normal['DATETIME'], normal['CPU_USAGE_AVG'], 'b-', alpha=0.7, label='정상')
    
    # 이상 데이터
    anomaly = sample_data[sample_data['is_anomaly']]
    ax.scatter(anomaly['DATETIME'], anomaly['CPU_USAGE_AVG'], c='red', s=50, label='이상', zorder=5)
    
    # 임계값 표시
    ax.axhline(y=80, color='orange', linestyle='--', alpha=0.7, label='경고 (80%)')
    ax.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='위험 (90%)')
    
    ax.set_xlabel('시간')
    ax.set_ylabel('CPU 사용률 (%)')
    ax.set_title(f'서버 {sample_mng_no} CPU 사용률 및 이상 탐지 결과')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### 6.2 NMS 네트워크 트래픽 이상 탐지

In [ ]:
# NMS IF 성능 이상 탐지 (멀티 피처)
print("=== NMS 네트워크 트래픽 이상 탐지 ===")

# 분석 대상 컬럼
traffic_cols = ['AVG_INBPS', 'AVG_OUTBPS', 'AVG_INERR', 'AVG_OUTERR']
available_traffic_cols = [c for c in traffic_cols if c in df_nms_if_perf.columns]

if len(available_traffic_cols) > 0:
    # 샘플링 (데이터가 크므로)
    df_sample = df_nms_if_perf.sample(n=min(50000, len(df_nms_if_perf)), random_state=42)
    
    # ML 기반 이상 탐지
    ml_detector = MLAnomalyDetector(contamination=0.05)
    df_traffic_anomaly = ml_detector.fit_predict(df_sample, available_traffic_cols)
    
    anomaly_count = df_traffic_anomaly['is_anomaly_ml'].sum()
    print(f"\n분석 데이터: {len(df_traffic_anomaly):,}건")
    print(f"ML 이상 탐지: {anomaly_count:,}건 ({anomaly_count/len(df_traffic_anomaly)*100:.2f}%)")
    
    # 이상 데이터 상위 10건
    print("\n=== 이상 심각도 상위 10건 ===")
    top_anomalies = df_traffic_anomaly[df_traffic_anomaly['is_anomaly_ml']].nsmallest(10, 'anomaly_score')
    display(top_anomalies[['DATETIME', 'MNG_NO', 'IF_IDX'] + available_traffic_cols + ['anomaly_score']])

### 6.3 복합 지표 이상 탐지 (CPU + 메모리 상관관계)

In [ ]:
# CPU와 메모리 데이터 병합
print("=== 복합 지표 이상 탐지 (CPU + 메모리) ===")

# 같은 시간대, 같은 서버의 CPU와 메모리 데이터 병합
df_combined = df_sms_cpu.merge(
    df_sms_memory[['YYYYMMDD', 'TIME_ID', 'MNG_NO', 'PHYSICAL_USED_PCT']],
    on=['YYYYMMDD', 'TIME_ID', 'MNG_NO'],
    how='inner'
)

print(f"병합된 데이터: {len(df_combined):,}건")

if len(df_combined) > 0 and 'CPU_USAGE_AVG' in df_combined.columns:
    # 복합 이상 탐지
    ml_detector = MLAnomalyDetector(contamination=0.05)
    df_combined_anomaly = ml_detector.fit_predict(df_combined, ['CPU_USAGE_AVG', 'PHYSICAL_USED_PCT'])
    
    # 시각화
    fig, ax = plt.subplots(figsize=(10, 8))
    
    normal = df_combined_anomaly[~df_combined_anomaly['is_anomaly_ml']]
    anomaly = df_combined_anomaly[df_combined_anomaly['is_anomaly_ml']]
    
    ax.scatter(normal['CPU_USAGE_AVG'], normal['PHYSICAL_USED_PCT'], 
               c='blue', alpha=0.3, s=10, label='정상')
    ax.scatter(anomaly['CPU_USAGE_AVG'], anomaly['PHYSICAL_USED_PCT'], 
               c='red', alpha=0.7, s=30, label='이상')
    
    ax.set_xlabel('CPU 사용률 (%)')
    ax.set_ylabel('메모리 사용률 (%)')
    ax.set_title('CPU vs 메모리 사용률 - 복합 이상 탐지')
    ax.legend()
    ax.axvline(x=80, color='orange', linestyle='--', alpha=0.5)
    ax.axhline(y=80, color='orange', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n복합 이상 탐지: {anomaly['is_anomaly_ml'].sum():,}건")

## 7. 이상 탐지 결과 요약 생성

In [ ]:
class AnomalySummarizer:
    """
    이상 탐지 결과를 LLM에 전달할 수 있는 형태로 요약
    """
    
    def __init__(self, device_info):
        self.device_info = device_info
    
    def summarize(self, anomaly_df, metric_name, time_col='DATETIME', 
                  device_col='MNG_NO', value_col=None, anomaly_col='is_anomaly'):
        """
        이상 탐지 결과 요약 생성
        """
        anomalies = anomaly_df[anomaly_df[anomaly_col]].copy()
        
        if len(anomalies) == 0:
            return {"status": "정상", "message": "이상 징후가 탐지되지 않았습니다."}
        
        # 장비 정보 병합
        anomalies = anomalies.merge(self.device_info, on=device_col, how='left')
        
        # 요약 생성
        summary = {
            "status": "이상",
            "metric": metric_name,
            "total_anomalies": len(anomalies),
            "affected_devices": anomalies[device_col].nunique(),
            "time_range": {
                "start": str(anomalies[time_col].min()),
                "end": str(anomalies[time_col].max())
            },
            "device_summary": [],
            "severity_distribution": {}
        }
        
        # 장비별 요약
        for mng_no in anomalies[device_col].unique()[:10]:  # 상위 10개 장비
            dev_anomalies = anomalies[anomalies[device_col] == mng_no]
            dev_info = dev_anomalies.iloc[0]
            
            device_summary = {
                "mng_no": int(mng_no),
                "dev_name": dev_info.get('DEV_NAME', 'Unknown'),
                "dev_ip": dev_info.get('DEV_IP', 'Unknown'),
                "anomaly_count": len(dev_anomalies),
            }
            
            if value_col and value_col in dev_anomalies.columns:
                device_summary["value_stats"] = {
                    "min": float(dev_anomalies[value_col].min()),
                    "max": float(dev_anomalies[value_col].max()),
                    "mean": float(dev_anomalies[value_col].mean())
                }
            
            summary["device_summary"].append(device_summary)
        
        # 심각도 분포 (있는 경우)
        if 'severity' in anomalies.columns:
            summary["severity_distribution"] = anomalies['severity'].value_counts().to_dict()
        
        return summary
    
    def generate_llm_prompt(self, summary):
        """
        LLM에 전달할 프롬프트 생성
        """
        if summary["status"] == "정상":
            return summary["message"]
        
        prompt = f"""
다음은 IT 인프라 모니터링 시스템에서 탐지된 이상 징후입니다. 분석하고 대응 방안을 제시해주세요.

## 탐지 개요
- 모니터링 지표: {summary['metric']}
- 총 이상 건수: {summary['total_anomalies']:,}건
- 영향 받은 장비 수: {summary['affected_devices']}대
- 탐지 기간: {summary['time_range']['start']} ~ {summary['time_range']['end']}

## 영향 받은 장비 상세
"""
        for dev in summary['device_summary']:
            prompt += f"\n### 장비: {dev['dev_name']} (IP: {dev['dev_ip']})\n"
            prompt += f"- 장비번호: {dev['mng_no']}\n"
            prompt += f"- 이상 발생 횟수: {dev['anomaly_count']}회\n"
            if 'value_stats' in dev:
                prompt += f"- 측정값 범위: {dev['value_stats']['min']:.2f} ~ {dev['value_stats']['max']:.2f} (평균: {dev['value_stats']['mean']:.2f})\n"
        
        if summary.get('severity_distribution'):
            prompt += f"\n## 심각도 분포\n"
            for sev, count in summary['severity_distribution'].items():
                prompt += f"- {sev}: {count}건\n"
        
        prompt += """
## 요청사항
1. 위 이상 징후의 가능한 원인을 분석해주세요.
2. 예상되는 영향 범위를 설명해주세요.
3. 권장하는 조치 사항을 우선순위와 함께 제시해주세요.
4. 향후 유사 상황을 예방하기 위한 방안을 제안해주세요.
"""
        return prompt

print("AnomalySummarizer 클래스 정의 완료!")

In [ ]:
# 이상 탐지 결과 요약 생성
summarizer = AnomalySummarizer(device_info_sms)

if 'is_anomaly' in df_cpu_anomaly.columns:
    cpu_summary = summarizer.summarize(
        df_cpu_anomaly, 
        metric_name="CPU 사용률",
        value_col='CPU_USAGE_AVG'
    )
    
    print("=== CPU 이상 탐지 요약 ===")
    import json
    print(json.dumps(cpu_summary, indent=2, ensure_ascii=False, default=str))

In [ ]:
# LLM 프롬프트 생성
if 'is_anomaly' in df_cpu_anomaly.columns:
    llm_prompt = summarizer.generate_llm_prompt(cpu_summary)
    print("=== LLM 분석 요청 프롬프트 ===")
    print(llm_prompt)

## 8. 로컬 LLM 연동 (Ollama)

In [ ]:
# 내부망 Ollama 연동 클래스
import requests
import json
import time
import os

class LocalLLMAnalyzer:
    """
    내부망 Ollama 서버를 이용한 LLM 분석기
    
    서버 정보:
    - HOST: http://210.107.60.21:11434
    - MODEL: gpt-oss:120b (SKT A.X-4.0-Light 기반)
    """
    
    # 내부망 서버 설정
    OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://210.107.60.21:11434")
    OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:120b")
    RETRIES = 3
    TEMPERATURE = 0.7
    
    def __init__(self, base_url=None, model=None):
        self.base_url = (base_url or self.OLLAMA_HOST).rstrip("/")
        self.model = model or self.OLLAMA_MODEL
    
    def check_connection(self):
        """Ollama 서버 연결 확인"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=10)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [m['name'] for m in models]
                print(f"✅ Ollama 연결 성공!")
                print(f"   서버: {self.base_url}")
                print(f"   사용 모델: {self.model}")
                print(f"   사용 가능 모델: {model_names[:5]}...")  # 상위 5개만 표시
                return True
        except Exception as e:
            print(f"❌ Ollama 연결 실패: {e}")
            print(f"   서버 주소: {self.base_url}")
            print("\n[내부망 접속 필요]")
            print("   이 서버는 내부망에서만 접속 가능합니다.")
        return False
    
    def analyze(self, prompt, system_prompt=None, temperature=None):
        """
        LLM에 분석 요청 (재시도 로직 포함)
        - /api/chat 우선 시도
        - 실패시 /api/generate로 fallback
        """
        if system_prompt is None:
            system_prompt = """당신은 IT 인프라 전문가입니다. 
네트워크 및 서버 모니터링 데이터를 분석하고, 이상 징후의 원인과 영향을 파악하며, 
적절한 대응 방안을 제시하는 역할을 합니다. 
분석 결과는 명확하고 실행 가능한 조언으로 제공해주세요.
한국어로 답변해주세요."""
        
        temp = temperature or self.TEMPERATURE
        last_err = None
        
        # 1차 시도: /api/chat 엔드포인트
        payload_chat = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            "options": {
                "temperature": temp,
                "num_ctx": 8192,
            },
            "stream": False,
        }
        
        for attempt in range(1, self.RETRIES + 1):
            try:
                print(f"   /api/chat 시도 {attempt}/{self.RETRIES}...")
                r = requests.post(
                    f"{self.base_url}/api/chat",
                    json=payload_chat,
                    timeout=180
                )
                r.raise_for_status()
                data = r.json()
                result = data.get("message", {}).get("content", "").strip()
                if result:
                    return result
            except Exception as e:
                last_err = e
                print(f"   시도 {attempt} 실패: {e}")
                time.sleep(0.4 * attempt)
        
        # 2차 시도: /api/generate 엔드포인트 (fallback)
        print("   /api/generate로 재시도...")
        formatted_prompt = f"### 시스템 지시 ###\n{system_prompt}\n\n### 사용자 입력 ###\n{prompt}\n"
        payload_gen = {
            "model": self.model,
            "prompt": formatted_prompt,
            "options": {
                "temperature": temp,
                "num_ctx": 8192,
            },
            "stream": False,
        }
        
        for attempt in range(1, self.RETRIES + 1):
            try:
                print(f"   /api/generate 시도 {attempt}/{self.RETRIES}...")
                r = requests.post(
                    f"{self.base_url}/api/generate",
                    json=payload_gen,
                    timeout=180
                )
                r.raise_for_status()
                data = r.json()
                result = data.get("response", "").strip()
                if result:
                    return result
            except Exception as e:
                last_err = e
                print(f"   시도 {attempt} 실패: {e}")
                time.sleep(0.4 * attempt)
        
        raise RuntimeError(f"Ollama 호출 실패: {last_err}")

print("LocalLLMAnalyzer 클래스 정의 완료!")
print(f"  - 서버: {LocalLLMAnalyzer.OLLAMA_HOST}")
print(f"  - 모델: {LocalLLMAnalyzer.OLLAMA_MODEL}")

In [ ]:
# LLM 연결 테스트 및 분석 실행
# 내부망 서버 사용 (http://210.107.60.21:11434)

llm = LocalLLMAnalyzer()  # 기본값: gpt-oss:120b 모델 사용

print("=== LLM 서버 연결 테스트 ===")
if llm.check_connection():
    print("\n=== LLM 분석 실행 중... (최대 3분 소요) ===")
    try:
        analysis_result = llm.analyze(llm_prompt)
        print("\n" + "="*60)
        print("LLM 분석 결과")
        print("="*60)
        print(analysis_result)
    except Exception as e:
        print(f"\n❌ 분석 실패: {e}")
else:
    print("\n⚠️ 내부망 Ollama 서버에 연결할 수 없습니다.")
    print("   - 내부망(사내 네트워크)에서만 접속 가능합니다.")
    print("   - 외부에서 접속하려면 ICT부서에 API 접속 권한을 요청하세요.")
    print("\n📋 위의 프롬프트를 직접 LLM에 입력하여 분석받을 수 있습니다.")

## 9. 결과 저장

In [ ]:
# 이상 탐지 결과 저장
OUTPUT_PATH = './output/anomaly_detection/'
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

# CPU 이상 탐지 결과 저장
if 'is_anomaly' in df_cpu_anomaly.columns:
    anomaly_only = df_cpu_anomaly[df_cpu_anomaly['is_anomaly']]
    anomaly_only.to_csv(f'{OUTPUT_PATH}cpu_anomalies.csv', index=False, encoding='utf-8-sig')
    print(f"CPU 이상 탐지 결과 저장: {len(anomaly_only)}건")

# 전체 통계 저장
if 'is_anomaly' in df_cpu_anomaly.columns:
    with open(f'{OUTPUT_PATH}analysis_summary.json', 'w', encoding='utf-8') as f:
        json.dump(cpu_summary, f, ensure_ascii=False, indent=2, default=str)
    print(f"분석 요약 저장 완료")

print(f"\n결과 저장 위치: {OUTPUT_PATH}")

---
## 다음 단계

1. **Streamlit 대시보드 구현** - 시각화 및 실시간 모니터링
2. **SFR-003 원인 분석** - 장비간 영향 관계 분석
3. **알림 시스템** - 이상 탐지 시 자동 알림